# 08 `GradScaler`：为什么 `scaler.step(optimizer)` 不是普通的 `step()`？

## 知识点标注
- 行业状态：常见 GPU 混合精度工程实践。
- 现实用途：在 float16 等低精度计算中，避免很小的梯度变成 0。
- 本课类型：概念演示，不使用 GPU，不训练 MokioMind。

先记住普通流程和混合精度流程的区别。

## 普通流程

```text
loss
  ↓
backward()       得到真实梯度
  ↓
clip_grad_norm_  可选：限制梯度整体长度
  ↓
optimizer.step() 修改参数
  ↓
zero_grad()      清空梯度
```

如果梯度太小，在低精度表示时可能被舍入成 0。

## 混合精度流程

```text
loss
  ↓ scale(loss)       先把 loss 放大
backward()            得到放大后的梯度
  ↓ unscale_()         更新前还原真实梯度
clip_grad_norm_       在真实尺度上裁剪
  ↓ scaler.step(...)   检查并执行参数更新
scaler.update()       调整下一轮缩放因子
zero_grad()           清空梯度
```

先解释两个术语：`inf` 是 infinity，表示数值大到超出当前数据类型能表示的范围；`nan` 是 Not a Number，表示某次运算已经产生了无效数字，例如 `0/0`。它们都不是正常的有限梯度。`scaler.step(optimizer)` 会先检查梯度里有没有 `inf` 或 `nan`。如果有，说明这轮数值已经坏掉，就跳过更新，避免把坏数值写进模型。

In [1]:
import torch

w = torch.tensor([1.0], requires_grad=True)
loss = (w - 3.0) ** 2
loss.backward()
print('普通 backward：loss =', loss.item(), '真实梯度 =', w.grad.item())

w.grad = None
scale = 1000.0
scaled_loss = ((w - 3.0) ** 2) * scale
scaled_loss.backward()
print('放大 loss 后：放大梯度 =', w.grad.item())
print('除以 scale 后：真实梯度 =', w.grad.item() / scale)

普通 backward：loss = 4.0 真实梯度 = -4.0
放大 loss 后：放大梯度 = -4000.0
除以 scale 后：真实梯度 = -4.0


### 先预测

1. 原始梯度是多少？
2. loss 放大 1000 倍后，梯度会放大多少倍？
3. `unscale_()` 的概念作用是什么？

## 为什么缩放因子不是固定的？

缩放因子是一个会根据训练情况变化的安全旋钮：

```text
梯度太小、没有溢出 -> 下一轮可以尝试放大更多
出现 inf（超出范围）或 nan（无效数字） -> 跳过本轮，并把缩放因子调小
```

目标是在不溢出的前提下尽量把小梯度放大，避免它们在低精度中变成 0。

## 溢出时到底发生什么？

```text
scale(loss)
    ↓
梯度变成 inf 或 nan
    ↓
scaler.step(optimizer) 发现坏数值
    ↓
跳过本次参数更新
    ↓
scaler.update() 减小下一轮 scale
```

## 对应 MokioMind 真实代码

```python
loss = loss / args.accumulation_steps
scaler.scale(loss).backward()

if step % args.accumulation_steps == 0:
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
```

逐行对应：

- [train_pretrain.py:63](/Users/mrwong/Documents/Program/AI/Minimind/eg/MokioMind-master/trainer/train_pretrain.py:63)：先处理梯度累积的平均 loss。
- [train_pretrain.py:65](/Users/mrwong/Documents/Program/AI/Minimind/eg/MokioMind-master/trainer/train_pretrain.py:65)：放大 loss 后反向传播。
- [train_pretrain.py:69](/Users/mrwong/Documents/Program/AI/Minimind/eg/MokioMind-master/trainer/train_pretrain.py:69)：把放大的梯度还原到真实尺度。
- [train_pretrain.py:71](/Users/mrwong/Documents/Program/AI/Minimind/eg/MokioMind-master/trainer/train_pretrain.py:71)：在真实尺度上做梯度裁剪。
- [train_pretrain.py:76](/Users/mrwong/Documents/Program/AI/Minimind/eg/MokioMind-master/trainer/train_pretrain.py:76)：检查溢出并执行参数更新。

## 验收题

1. `scaler.scale(loss)` 改变的是模型参数、loss 数值，还是 tokenizer？
2. 为什么裁剪前要先 `scaler.unscale_(optimizer)`？
3. `scaler.step(optimizer)` 与普通 `optimizer.step()` 的共同点和额外作用分别是什么？
4. `scaler.update()` 和 `optimizer.zero_grad()` 分别管理什么？